# Chapter 8: Kernel Methods

> You don't need to expand your features to use them — you just need to know how to take their dot product.

**Type:** Learn + Build &nbsp;|&nbsp; **Language:** Python &nbsp;|&nbsp; **Prerequisites:** Chapter 3 (The Perceptron) &nbsp;|&nbsp; **Time:** ~45 minutes
**Source:** *A Course in Machine Learning*, Hal Daumé III — Chapter 9

---

## Learning Objectives

- Explain how the kernel trick avoids explicitly computing an exploded feature space
- Implement linear, polynomial, and RBF kernels from scratch
- Implement the kernelized perceptron (representer theorem in action) and extend it to multi-class problems via one-vs-rest
- Compare kernelized models against scikit-learn's SVC (also a kernel machine) on real data
- Explain the connection between "confusable" training points and support vectors

## The Problem

Linear models are convex and easy to optimize, but they can only express linear decision boundaries. Chapter 4 showed you can get around this by exploding your feature space (e.g., adding all pairwise products), but this is computationally prohibitive once you have more than a few hundred features — a quadratic expansion alone squares both your memory and (roughly) your data requirements.

The kernel trick asks: can we get the benefit of this feature explosion without ever actually constructing the expanded vectors?

## The Concept

```
Training examples x --> Kernel K(x, z) = phi(x) . phi(z) --> Perceptron / SVM update rule
   --> Weight vector w = sum of alpha_n * phi(x_n)
   --> Prediction uses only K(x_train, x_test), never phi() directly
```

### Key Ideas

- **Representer theorem**: throughout training, the perceptron's weight vector is always a linear combination of the (mapped) training examples, so every computation — training and prediction — can be rewritten purely in terms of dot products between examples
- **A kernel is just a generalized dot product**: `K(x, z) = φ(x) · φ(z)`. Polynomial kernels `(1 + x·z)^d` correspond to feature expansions with all degree-`d` combinations of features; RBF kernels `exp(-γ‖x-z‖²)` behave like an infinite-dimensional feature expansion and act like a Gaussian "vote" from every nearby training point
- **Same cost, more power**: computing a degree-3 polynomial kernel costs exactly the same as computing a plain dot product (plus one add and one power) — you get cubic feature interactions "for free"
- **Only some points matter**: the kernel perceptron (like the SVM) only updates on points it currently misclassifies. Points that are never confusable with the wrong class end up with zero weight — conceptually the same idea as support vectors in an SVM (Section 9.6)

## Build It

### Setup

We'll need NumPy for array operations, and several utilities from scikit-learn: the Wine and Digits datasets, a train/test splitter, a feature scaler, scikit-learn's own `SVC` (also a kernel machine, used as a sanity check), and an accuracy metric.

In [1]:
import numpy as np
from sklearn.datasets import load_wine, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

RNG = np.random.RandomState(0)

### Step 1: Kernels as Drop-In Replacements for the Dot Product (Eq 9.13, 9.18)

A kernel is just a function `K(x, z)` that behaves like a dot product in some (possibly much higher-dimensional) feature space. The three kernels below cover the linear case (no expansion at all), the polynomial case (all degree-`d` feature interactions), and the RBF case (an implicit infinite-dimensional expansion).

In [2]:
def linear_kernel(X, Z):
    return X @ Z.T


def polynomial_kernel(X, Z, degree=3):
    return (1.0 + X @ Z.T) ** degree


def rbf_kernel(X, Z, gamma=0.05):
    sq_x = np.sum(X ** 2, axis=1)[:, None]
    sq_z = np.sum(Z ** 2, axis=1)[None, :]
    sq_dists = sq_x + sq_z - 2 * X @ Z.T
    sq_dists = np.maximum(sq_dists, 0.0)
    return np.exp(-gamma * sq_dists)


KERNELS = {
    "linear": linear_kernel,
    "poly": lambda X, Z: polynomial_kernel(X, Z, degree=3),
    "rbf": lambda X, Z: rbf_kernel(X, Z, gamma=0.05),
}

### Step 2: The Kernelized Perceptron (Algorithm 9.2)

This is the representer theorem in action: notice that the weight vector `w` never appears explicitly anywhere in this class. Training and prediction only ever touch the data through kernel values — the full `n x n` Gram matrix `K` is precomputed once, and the activation for example `i` is just a weighted sum of kernel values against every other example. On a mistake, the mistake-driven update only touches `alpha[i]` and `b`, exactly as in the plain perceptron (Chapter 3), but expressed through the kernel instead of a raw dot product.

In [3]:
class KernelPerceptronFromScratch:
    def __init__(self, kernel="rbf", n_iter=20):
        self.kernel_name = kernel
        self.kernel_fn = KERNELS[kernel]
        self.n_iter = n_iter

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y, dtype=float)
        n = X.shape[0]
        self.X_train = X
        self.alpha = np.zeros(n)
        self.b = 0.0

        K = self.kernel_fn(X, X)

        for _ in range(self.n_iter):
            for i in range(n):
                a = np.sum(self.alpha * y * K[i]) + self.b
                if y[i] * a <= 0:
                    self.alpha[i] += 1
                    self.b += y[i]

        self._coef = self.alpha * y
        return self

    def decision_function(self, X):
        X = np.asarray(X)
        K = self.kernel_fn(X, self.X_train)
        return K @ self._coef + self.b

    def predict(self, X):
        return np.where(self.decision_function(X) >= 0, 1, -1)

### Step 3: One-vs-Rest Wrapper for Multi-Class Problems (Section 5.2, OVA)

The kernel perceptron above is inherently binary. To handle the multi-class Wine dataset, we reuse Chapter 5's one-vs-all reduction: train one binary kernel perceptron per class (that class vs. everything else), then predict whichever class's model produces the highest score.

In [4]:
class OneVsRestKernelPerceptron:
    def __init__(self, kernel="rbf", n_iter=20):
        self.kernel = kernel
        self.n_iter = n_iter

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.models_ = {}
        for c in self.classes_:
            y_bin = np.where(y == c, 1, -1)
            self.models_[c] = KernelPerceptronFromScratch(
                kernel=self.kernel, n_iter=self.n_iter
            ).fit(X, y_bin)
        return self

    def predict(self, X):
        scores = np.column_stack(
            [self.models_[c].decision_function(X) for c in self.classes_]
        )
        return self.classes_[np.argmax(scores, axis=1)]

## Use It — Real Data

### Experiment A: Wine Dataset — Linear vs. Polynomial vs. RBF Kernel

The **Wine** dataset is a real, three-class chemical-analysis benchmark bundled with scikit-learn. Features are standardized (zero mean, unit variance) before training, and we compare all three kernels through the one-vs-rest wrapper above.

In [5]:
wine = load_wine()
Xw, yw = wine.data, wine.target
print(f"Dataset shape: {Xw.shape[0]} examples, {Xw.shape[1]} features, {len(set(yw))} classes")

Xw_train, Xw_test, yw_train, yw_test = train_test_split(
    Xw, yw, test_size=0.3, random_state=42, stratify=yw
)
scaler = StandardScaler().fit(Xw_train)
Xw_train_s = scaler.transform(Xw_train)
Xw_test_s = scaler.transform(Xw_test)

print(f"\n{'kernel':>8} | {'train_acc':>10} | {'test_acc':>9}")
print("-" * 34)
for kname in ["linear", "poly", "rbf"]:
    model = OneVsRestKernelPerceptron(kernel=kname, n_iter=20).fit(Xw_train_s, yw_train)
    train_acc = accuracy_score(yw_train, model.predict(Xw_train_s))
    test_acc = accuracy_score(yw_test, model.predict(Xw_test_s))
    print(f"{kname:>8} | {train_acc:>10.4f} | {test_acc:>9.4f}")

Dataset shape: 178 examples, 13 features, 3 classes

  kernel |  train_acc |  test_acc
----------------------------------
  linear |     1.0000 |    0.9444
    poly |     1.0000 |    0.9259
     rbf |     1.0000 |    0.9815


### Sanity Check Against `sklearn.svm.SVC`

`SVC` is also a kernel machine (it solves the SVM dual rather than the plain perceptron), so it's a natural point of comparison. We don't expect identical numbers — the SVM optimizes a regularized margin objective, while our perceptron just hunts for *any* separating combination — but the two independent implementations should broadly agree on which kernel performs best.

In [6]:
print("--- Sanity check vs sklearn.svm.SVC (also a kernel machine) ---")
for kname, sk_kernel in [("linear", "linear"), ("poly", "poly"), ("rbf", "rbf")]:
    svc = SVC(kernel=sk_kernel, degree=3, gamma=0.05 if sk_kernel == "rbf" else "scale")
    svc.fit(Xw_train_s, yw_train)
    acc = accuracy_score(yw_test, svc.predict(Xw_test_s))
    print(f"SVC(kernel={sk_kernel:<7}) test acc: {acc:.4f}")

--- Sanity check vs sklearn.svm.SVC (also a kernel machine) ---
SVC(kernel=linear ) test acc: 0.9630
SVC(kernel=poly   ) test acc: 0.9074
SVC(kernel=rbf    ) test acc: 1.0000


### Experiment B: Digit '3' vs. '8' — a Genuinely Non-Linear Boundary

Handwritten digits **3** and **8** are visually similar and notoriously hard to separate with a straight line. This is a binary problem, so it uses `KernelPerceptronFromScratch` directly (no one-vs-rest wrapper needed), and it's a good stress test for whether the polynomial and RBF kernels actually earn their keep over the plain linear kernel.

In [7]:
digits = load_digits()
mask = np.isin(digits.target, [3, 8])
Xd, yd_raw = digits.data[mask], digits.target[mask]
yd = np.where(yd_raw == 3, 1, -1)
print(f"Dataset shape: {Xd.shape[0]} examples, {Xd.shape[1]} pixel features (8x8 images)")

Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    Xd, yd, test_size=0.3, random_state=42, stratify=yd
)
scaler_d = StandardScaler().fit(Xd_train)
Xd_train_s = scaler_d.transform(Xd_train)
Xd_test_s = scaler_d.transform(Xd_test)

print(f"\n{'kernel':>8} | {'train_acc':>10} | {'test_acc':>9}")
print("-" * 34)
for kname in ["linear", "poly", "rbf"]:
    model = KernelPerceptronFromScratch(kernel=kname, n_iter=20).fit(Xd_train_s, yd_train)
    train_acc = accuracy_score(yd_train, model.predict(Xd_train_s))
    test_acc = accuracy_score(yd_test, model.predict(Xd_test_s))
    print(f"{kname:>8} | {train_acc:>10.4f} | {test_acc:>9.4f}")

Dataset shape: 357 examples, 64 pixel features (8x8 images)

  kernel |  train_acc |  test_acc
----------------------------------
  linear |     1.0000 |    0.9815
    poly |     1.0000 |    0.9815


     rbf |     1.0000 |    0.9630


### Section 9.6: How Many Training Points Actually Matter?

The representer theorem says `w` is a combination of *some* training examples — but not necessarily all of them. Points the perceptron never mistakes end up with `alpha = 0` and contribute nothing to the final decision function. This is exactly the support-vector intuition from the SVM literature: only "confusable" points near the boundary end up mattering.

In [8]:
rbf_model = KernelPerceptronFromScratch(kernel="rbf", n_iter=20).fit(Xd_train_s, yd_train)
n_touched = np.sum(rbf_model.alpha > 0)
print(f"Training examples that ever triggered a perceptron update: "
      f"{n_touched} / {len(yd_train)} ({100 * n_touched / len(yd_train):.1f}%)")

Training examples that ever triggered a perceptron update: 24 / 249 (9.6%)


**Reading the result:** the RBF kernel gives the best test accuracy for both the from-scratch kernel perceptron and scikit-learn's SVC on the same data — the two independent implementations agree on which kernel wins, which is a good correctness signal. And only a small fraction of training examples ever triggered a weight update, echoing the support-vector intuition that only points near the decision boundary matter.

## When to Use Kernel Methods

| API / Function | When to use it |
|---|---|
| `KernelPerceptronFromScratch(kernel="rbf").fit(X, y)` | Binary classification where you suspect a non-linear boundary and want an easy-to-inspect model |
| `OneVsRestKernelPerceptron` | Multi-class extension via Chapter 5's OVA reduction |
| `polynomial_kernel(X, Z, degree=d)` | When you believe interactions of exactly `d` features matter (e.g., pairwise feature interactions at `d=2`) |
| `rbf_kernel(X, Z, gamma=...)` | Default choice when you have no strong prior about the shape of the decision boundary; tune `gamma` on held-out data |
| `sklearn.svm.SVC(kernel=...)` | Production use — solves the (much better regularized) SVM dual rather than the plain perceptron |

## Exercises

1. Implement the kernelized K-means algorithm from Section 9.3 and cluster the Wine dataset using the RBF kernel; compare cluster purity to plain K-means from Chapter 2.
2. Show empirically that `K(x, z) = (1 + x·z)^2` gives the same result as explicitly expanding `x` into all products of pairs of features and taking the plain dot product.
3. Vary `gamma` in the RBF kernel from very small to very large and observe the train/test accuracy trend — relate this to underfitting/overfitting.

## Key Terms

| Term | Common Assumption | Precise Meaning |
|---|---|---|
| **Kernel** | "A fancy similarity score" | A function `K(x,z)` that is guaranteed to equal an inner product `φ(x)·φ(z)` for *some* (possibly infinite-dimensional) feature mapping φ |
| **Kernel Trick** | "A speed optimization" | A re-derivation of a learning algorithm so it only ever needs pairwise kernel values, never the explicit feature mapping — this is what makes infinite-dimensional feature spaces usable at all |
| **Support Vector** | "Any training point" | A training point whose associated dual weight is non-zero at the optimum — geometrically, one that lies on or inside the margin |
| **Gram Matrix** | "Just a distance matrix" | The `N×N` matrix of all pairwise kernel values `K(x_i, x_j)`, which is all a kernel algorithm ever needs from the training data |

## Summary

- The **kernel trick** lets a linear-style learner benefit from an exploded (even infinite-dimensional) feature space, without ever constructing the expanded vectors — everything is rewritten in terms of pairwise kernel values
- The **representer theorem** guarantees the weight vector is always a linear combination of (mapped) training examples, which is exactly why kernelization is possible
- **Linear, polynomial, and RBF kernels** trade off simplicity for representational power at essentially the same computational cost as a plain dot product
- Only a small fraction of training points ever trigger a perceptron update — the same "confusable points near the boundary" intuition behind support vectors in an SVM
- Our from-scratch kernel perceptron agrees with scikit-learn's `SVC` on which kernel performs best, on two independent real datasets

---

**Next:** Chapter 9 — beyond kernels